In [0]:
looktables_rules = {
    "rule1" : "show_id is NOT NULL",
}

# 1. Create streaming on top of every table in silver

### This is delta live table
It automatically creates the table and incrementally reads table from the location given

In [0]:
# decorator
@dlt.table(
    # give name to the table
    name = "gold_netflixdirectors"
)


# looktables_rules is a dictionary. If any of the rules fail, the table is dropped
@dlt.expect_all_or_drop(looktables_rules)
def myfunc(): 
    # create a dataframe
    # readStream: used because this is a streaming table
    # Give location from where data to be read
    df = spark.readStream.format("delta").load("abfss://silver@netflixprojectdlansh.dfs.core.windows.net/netflix_directors")
    return df 

In [0]:
@dlt.table(
    name = "gold_netflixcast"
)

@dlt.expect_all_or_drop(looktables_rules)
def myfunc(): 
    df = spark.readStream.format("delta").load("abfss://silver@netflixprojectdlansh.dfs.core.windows.net/netflix_cast")
    return df 


In [0]:
@dlt.table(
    name = "gold_netflixcountries"
)

@dlt.expect_all_or_drop(looktables_rules)
def myfunc(): 
    df = spark.readStream.format("delta").load("abfss://silver@netflixprojectdlansh.dfs.core.windows.net/netflix_countries")
    return df 


In [0]:
@dlt.table(
    name = "gold_netflixcategory"
)

@dlt.expect_or_drop("rule1","show_id is NOT NULL")
def myfunc(): 
    df = spark.readStream.format("delta").load("abfss://silver@netflixprojectdlansh.dfs.core.windows.net/netflix_category")
    return df 


All 4 DLT are created

All the tables have a common column "show_id"
Creating expectation/rule to apply on all the tables

# 2. CREATE EXPECTATION

### 2.1 Create Dictionary
(cell 1)

In [0]:
@dlt.table 

def gold_stg_netflixtitles():

    df = spark.readStream.format("delta").load("abfss://silver@netflixprojectdlansh.dfs.core.windows.net/netflix_titles")
    return df

In [0]:
from pyspark.sql.functions import *

Creating a view

In [0]:
@dlt.view

def gold_trns_netflixtitles():
    df = spark.readStream.table("LIVE.gold_stg_netflixtitles")
    df = df.withColumn("newflag",lit(1))
    return df 

In [0]:
masterdata_rules = {
    "rule1" : "newflag is NOT NULL",
    "rule2" : "show_id is NOT NULL"
}

In [0]:
@dlt.table

@dlt.expect_all_or_drop(masterdata_rules)
def gold_netflixtitles():

    df = spark.readStream.table("LIVE.gold_trns_netflixtitles")

    return df